In [156]:
import pandas as pd

train_df = pd.read_parquet("../data/processed/train_pre_impute.parquet")
valid_df = pd.read_parquet("../data/processed/valid_pre_impute.parquet")
test_df = pd.read_parquet("../data/processed/test_pre_impute.parquet")

In [157]:
print(train_df.shape, valid_df.shape, test_df.shape)

(560, 22) (120, 22) (120, 22)


In [158]:
train_df.isnull().sum()

Age                        0
Year_of_Study              0
Department                13
Residence_Type            12
Attendance_Percent        36
Study_Hours_Per_Day       24
Previous_GPA              26
Backlogs                  11
Sleep_Hours               24
Screen_Time_Hours         22
Exercise_Freq_Per_Week     0
Social_Activity_Score     32
Part_Time_Job             16
Family_Income_Bracket      7
Financial_Stress_Score    30
Family_Support_Score      33
Stress_Level              35
Anxiety_Score             28
Motivation_Score          32
Peer_Pressure_Score       30
Counseling_Access          0
Dropout_Risk               0
dtype: int64

In [159]:
cols_to_check = [
    "Attendance_Percent",
    "Study_Hours_Per_Day",
    "Previous_GPA",
    "Backlogs",
    "Sleep_Hours",
    "Screen_Time_Hours",
    "Social_Activity_Score",
    "Financial_Stress_Score",
    "Family_Support_Score",
    "Stress_Level",
    "Anxiety_Score",
    "Motivation_Score",
    "Peer_Pressure_Score",
]

In [160]:
train_df[cols_to_check].skew()

Attendance_Percent       -0.245026
Study_Hours_Per_Day       0.161590
Previous_GPA             -0.000954
Backlogs                  1.470276
Sleep_Hours              -0.051310
Screen_Time_Hours         0.004666
Social_Activity_Score     0.033196
Financial_Stress_Score   -0.012787
Family_Support_Score     -0.245835
Stress_Level             -0.077685
Anxiety_Score            -0.034004
Motivation_Score          0.099014
Peer_Pressure_Score      -0.015224
dtype: float64

In [161]:
from sklearn.impute import SimpleImputer
mean_cols = [col for col in cols_to_check if col != "Backlogs"]
mean_imputer = SimpleImputer(strategy="mean")
mean_imputer.fit(train_df[mean_cols])

train_df[mean_cols] = mean_imputer.transform(train_df[mean_cols])
valid_df[mean_cols] = mean_imputer.transform(valid_df[mean_cols])
test_df[mean_cols] = mean_imputer.transform(test_df[mean_cols])

In [162]:
train_df[mean_cols].isnull().sum().sum()
valid_df[mean_cols].isnull().sum().sum()
test_df[mean_cols].isnull().sum().sum()

0

In [163]:
from sklearn.impute import SimpleImputer
median_cols = ["Backlogs"]
median_imputer = SimpleImputer(strategy="median")
median_imputer.fit(train_df[median_cols])

train_df[median_cols] = median_imputer.transform(train_df[median_cols])
valid_df[median_cols] = median_imputer.transform(valid_df[median_cols])
test_df[median_cols] = median_imputer.transform(test_df[median_cols])

In [164]:
print(train_df[median_cols].isnull().sum().sum())
print(valid_df[median_cols].isnull().sum().sum())
print(test_df[median_cols].isnull().sum().sum())

0
0
0


In [165]:
cat_cols = ["Department", "Residence_Type", "Part_Time_Job", "Family_Income_Bracket"]
cat_imputer = SimpleImputer(strategy="constant", fill_value="Unknown", missing_values=None)
cat_imputer.fit(train_df[cat_cols])

train_df[cat_cols] = cat_imputer.transform(train_df[cat_cols])
valid_df[cat_cols] = cat_imputer.transform(valid_df[cat_cols])
test_df[cat_cols] = cat_imputer.transform(test_df[cat_cols])

In [166]:
print(train_df.isnull().sum().sum())
print(valid_df.isnull().sum().sum())
print(test_df.isnull().sum().sum())

0
0
0


In [167]:
from sklearn.preprocessing import OneHotEncoder

unordered_cols= ["Department", "Residence_Type", "Part_Time_Job"]
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

encoder.fit(train_df[unordered_cols])

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [168]:
train_cat = encoder.transform(train_df[unordered_cols])
valid_cat = encoder.transform(valid_df[unordered_cols])
test_cat = encoder.transform(test_df[unordered_cols])
print(train_cat.shape)

(560, 14)


In [169]:
income_order = {"Low": 0, "Lower-Middle": 1, "Middle": 2, "Upper-Middle": 3, "High": 4, "Unknown": -1}

train_df["Family_Income_Bracket"] = train_df["Family_Income_Bracket"].map(income_order)
valid_df["Family_Income_Bracket"] = valid_df["Family_Income_Bracket"].map(income_order)
test_df["Family_Income_Bracket"] = test_df["Family_Income_Bracket"].map(income_order)

In [170]:
train_df["Counseling_Access"] = train_df["Counseling_Access"].map({"Yes": 1, "No": 0})
valid_df["Counseling_Access"] = valid_df["Counseling_Access"].map({"Yes": 1, "No": 0})
test_df["Counseling_Access"] = test_df["Counseling_Access"].map({"Yes": 1, "No": 0})

In [171]:
train_df["Family_Income_Bracket"].unique()

array([ 3,  1,  4,  2,  0, -1])

In [172]:
print(train_df.columns.tolist())

['Age', 'Year_of_Study', 'Department', 'Residence_Type', 'Attendance_Percent', 'Study_Hours_Per_Day', 'Previous_GPA', 'Backlogs', 'Sleep_Hours', 'Screen_Time_Hours', 'Exercise_Freq_Per_Week', 'Social_Activity_Score', 'Part_Time_Job', 'Family_Income_Bracket', 'Financial_Stress_Score', 'Family_Support_Score', 'Stress_Level', 'Anxiety_Score', 'Motivation_Score', 'Peer_Pressure_Score', 'Counseling_Access', 'Dropout_Risk']


In [173]:
cols_to_scale = [col for col in train_df.columns if col not in unordered_cols + ["Dropout_Risk"]]
print(len(cols_to_scale))
print(cols_to_scale)

18
['Age', 'Year_of_Study', 'Attendance_Percent', 'Study_Hours_Per_Day', 'Previous_GPA', 'Backlogs', 'Sleep_Hours', 'Screen_Time_Hours', 'Exercise_Freq_Per_Week', 'Social_Activity_Score', 'Family_Income_Bracket', 'Financial_Stress_Score', 'Family_Support_Score', 'Stress_Level', 'Anxiety_Score', 'Motivation_Score', 'Peer_Pressure_Score', 'Counseling_Access']


In [174]:
train_df["Counseling_Access"].dtype
train_df["Counseling_Access"].unique()

array([0, 1])

In [175]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_df[cols_to_scale])

train_df[cols_to_scale] = scaler.transform(train_df[cols_to_scale])
valid_df[cols_to_scale] = scaler.transform(valid_df[cols_to_scale])
test_df[cols_to_scale] = scaler.transform(test_df[cols_to_scale])

In [176]:
train_df[cols_to_scale].describe()

,Age,Year_of_Study,Attendance_Percent,Study_Hours_Per_Day,Previous_GPA,Backlogs,Sleep_Hours,Screen_Time_Hours,Exercise_Freq_Per_Week,Social_Activity_Score,Family_Income_Bracket,Financial_Stress_Score,Family_Support_Score,Stress_Level,Anxiety_Score,Motivation_Score,Peer_Pressure_Score,Counseling_Access
count,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02,5.600000e+02
mean,4.916702e-16,9.198991e-17,9.833404e-16,-5.392512e-17,3.425831e-16,5.075305e-17,3.298948e-16,-2.125284e-16,-7.612958e-17,2.093563e-16,-1.268826e-16,1.110223e-17,-3.806479e-17,-2.030122e-16,-1.490871e-16,-2.537653e-16,-5.107026e-16,5.075305e-17
std,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00,1.000894e+00
min,-1.542723e+00,-1.163245e+00,-3.062397e+00,-2.106974e+00,-3.109494e+00,-8.951801e-01,-2.840611e+00,-2.355186e+00,-1.332337e+00,-2.655100e+00,-2.335857e+00,-2.255188e+00,-3.176049e+00,-2.902726e+00,-2.504430e+00,-2.670968e+00,-2.474841e+00,-6.910662e-01
25%,-7.706724e-01,-1.163245e+00,-6.773138e-01,-6.750428e-01,-6.748029e-01,-8.951801e-01,-6.695853e-01,-6.508352e-01,-6.761286e-01,-6.161733e-01,-6.976854e-01,-6.359133e-01,-6.301135e-01,-6.192455e-01,-6.595441e-01,-6.385200e-01,-6.910201e-01,-6.910662e-01
50%,1.378663e-03,-2.534445e-01,1.070058e-15,0.000000e+00,6.236308e-16,1.513770e-01,8.860297e-03,4.204909e-16,-1.992060e-02,0.000000e+00,1.214002e-01,0.000000e+00,-4.433813e-16,-4.716600e-16,0.000000e+00,0.000000e+00,-4.951097e-16,-6.910662e-01
75%,7.734297e-01,6.563564e-01,7.571246e-01,6.918005e-01,6.417205e-01,1.513770e-01,6.194613e-01,6.392634e-01,6.362874e-01,6.908310e-01,9.404857e-01,7.134824e-01,6.178939e-01,6.154270e-01,6.368626e-01,6.503497e-01,6.468453e-01,1.447039e+00
max,1.545481e+00,1.566157e+00,1.726594e+00,3.034961e+00,2.212772e+00,5.384163e+00,2.586953e+00,3.089267e+00,3.261119e+00,2.572917e+00,1.759571e+00,2.242798e+00,1.815981e+00,2.354589e+00,2.481749e+00,2.286223e+00,2.820877e+00,1.447039e+00


In [177]:
cat_col_names = encoder.get_feature_names_out(unordered_cols)
print(cat_col_names)

['Department_Arts' 'Department_Business' 'Department_Engineering'
 'Department_Law' 'Department_Medicine' 'Department_Science'
 'Department_Unknown' 'Residence_Type_Day Scholar' 'Residence_Type_Hostel'
 'Residence_Type_PG/Rented' 'Residence_Type_Unknown' 'Part_Time_Job_No'
 'Part_Time_Job_Unknown' 'Part_Time_Job_Yes']


In [178]:
train_cat_df = pd.DataFrame(train_cat, columns=cat_col_names, index=train_df.index)
valid_cat_df = pd.DataFrame(valid_cat, columns=cat_col_names, index=valid_df.index)
test_cat_df = pd.DataFrame(test_cat, columns=cat_col_names, index=test_df.index)

In [179]:
train_cat_df.head()

,Department_Arts,Department_Business,Department_Engineering,Department_Law,Department_Medicine,Department_Science,Department_Unknown,Residence_Type_Day Scholar,Residence_Type_Hostel,Residence_Type_PG/Rented,Residence_Type_Unknown,Part_Time_Job_No,Part_Time_Job_Unknown,Part_Time_Job_Yes
0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0


In [180]:
train_df = train_df.drop(columns=unordered_cols)
train_df = pd.concat([train_df, train_cat_df], axis=1)

valid_df = valid_df.drop(columns=unordered_cols)
valid_df = pd.concat([valid_df, valid_cat_df], axis=1)

test_df = test_df.drop(columns=unordered_cols)
test_df = pd.concat([test_df, test_cat_df], axis=1)

In [181]:
print(train_df.shape, valid_df.shape, test_df.shape)

(560, 33) (120, 33) (120, 33)


In [182]:
print(train_df.isnull().sum().sum())
print(valid_df.isnull().sum().sum())
print(test_df.isnull().sum().sum())

0
0
0


In [184]:
train_df.to_parquet("../data/processed/train.parquet", index=False)
valid_df.to_parquet("../data/processed/valid.parquet", index=False)
test_df.to_parquet("../data/processed/test.parquet", index=False)